In [0]:

-- ============================================================
-- FRAUD DETECTION LAKEHOUSE
-- SQL ANALYTICS
-- ============================================================
-- Purpose:
-- Generate business-ready fraud analytics using the
-- Bronze-Silver-Gold Medallion Architecture.
--
-- Data sources:
--   Silver: fraud_detection.silver.transactions
--   Gold:   fraud_detection.gold.*
--
-- Sections:
--   1. Overall fraud KPIs (portfolio level)
--   2. Customer risk distribution (HIGH / MEDIUM / LOW)
--   3. Fraud by merchant category
--   4. Fraud by transaction country
--   5. Daily fraud trends over time
-- ============================================================


-- ============================================================
-- 1. OVERALL FRAUD KPIs
-- ============================================================
-- Portfolio-level health check computed directly from the
-- typed, deduplicated silver transactions table:
--   - fraud_rate_percentage = fraudulent count / total count
--   - fraudulent_transaction_amount uses a CASE filter so
--     only rows flagged fraud_label = 1 contribute to the sum
-- ============================================================

SELECT
    COUNT(*) AS total_transactions,

    SUM(fraud_label) AS fraudulent_transactions,

    ROUND(
        SUM(fraud_label) * 100.0 / COUNT(*),
        2
    ) AS fraud_rate_percentage,

    ROUND(
        SUM(amount),
        2
    ) AS total_transaction_amount,

    ROUND(
        SUM(
            CASE
                WHEN fraud_label = 1 THEN amount
                ELSE 0
            END
        ),
        2
    ) AS fraudulent_transaction_amount

FROM fraud_detection.silver.transactions;


-- ============================================================
-- 2. CUSTOMER RISK DISTRIBUTION
-- ============================================================
-- How customers are spread across the rule-based risk
-- categories assigned in 04_gold_aggregation:
--   - percentage_of_customers uses a windowed SUM over the
--     whole result set (SUM(COUNT(*)) OVER ()) so each row's
--     share is relative to all customers, not its group
--   - ORDER BY maps HIGH first so the riskiest segment
--     appears at the top instead of alphabetically
-- ============================================================

SELECT
    risk_category,

    COUNT(*) AS customer_count,

    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        2
    ) AS percentage_of_customers

FROM fraud_detection.gold.customer_risk_metrics

GROUP BY risk_category

ORDER BY
    CASE risk_category
        WHEN 'HIGH' THEN 1
        WHEN 'MEDIUM' THEN 2
        WHEN 'LOW' THEN 3
    END;


-- ============================================================
-- 3. FRAUD BY MERCHANT CATEGORY
-- ============================================================
-- Reads the pre-aggregated gold summary to avoid re-scanning
-- silver per category. fraud_rate is stored as a ratio (0-1),
-- so it is multiplied by 100 here for display. Categories are
-- ordered by fraud rate, worst first.
-- ============================================================

SELECT
    category,

    total_transactions,

    fraudulent_transactions,

    ROUND(
        fraud_rate * 100,
        2
    ) AS fraud_rate_percentage,

    total_transaction_amount,

    average_transaction_amount

FROM fraud_detection.gold.fraud_summary_by_category

ORDER BY fraud_rate DESC;


-- ============================================================
-- 4. FRAUD BY TRANSACTION COUNTRY
-- ============================================================
-- Country-level view from the gold summary, ordered by fraud
-- rate to surface the riskiest geographies first. The five
-- allowed countries (PH/US/SG/AU/GB) are enforced upstream
-- in silver, so no country filter is needed here.
-- ============================================================

SELECT
    country,

    total_transactions,

    fraudulent_transactions,

    ROUND(
        fraud_rate * 100,
        2
    ) AS fraud_rate_percentage,

    total_transaction_amount,

    fraudulent_transaction_amount

FROM fraud_detection.gold.fraud_summary_by_country

ORDER BY fraud_rate DESC;


-- ============================================================
-- 5. DAILY FRAUD TRENDS
-- ============================================================
-- Time-series view from the gold daily metrics table for
-- spotting spikes in fraud rate or volume. Ordered
-- chronologically so the series can be charted as-is.
-- ============================================================

SELECT
    transaction_date,

    total_transactions,

    fraudulent_transactions,

    ROUND(
        fraud_rate * 100,
        2
    ) AS fraud_rate_percentage,

    total_transaction_amount,

    average_transaction_amount,

    fraudulent_transaction_amount

FROM fraud_detection.gold.daily_transaction_metrics

ORDER BY transaction_date;
